# Eksplorimi i Dataset-it (EDA)

Dataset: Kosovo News Articles Dataset (Kaggle)

## 1. Leximi fillestar dhe struktura e dataset-it

In [11]:
import pandas as pd

# Lexo VETËM 5 rreshtat e parë, jo krejt file-in
df_sample = pd.read_csv('../data/raw/Kosovo-News-Articles.csv', nrows=5)

print("Kolonat:", df_sample.columns.tolist())
print("\nShembuj:")
df_sample

Kolonat: ['content', 'date', 'title', 'category', 'author', 'source', 'fetched_at']

Shembuj:


,content,date,title,category,author,source,fetched_at
0,Sed ut perspiciatis unde omnis iste natus erro...,2018-09-26 13:24:25,Si t’i kërkosh të dashurit se ka ardhur koha p...,Life &amp; Style,ernis zejnullahu,Ballkani,2024-11-12T09:49:10Z
1,Sed ut perspiciatis unde omnis iste natus erro...,2018-09-26 13:26:04,Arsyeja e saktë përse disa meshkuj janë beqarë...,Life &amp; Style,ernis zejnullahu,Ballkani,2024-11-12T09:49:10Z
2,Puna intensive është veçanërisht e rëndësishme...,2018-09-26 13:44:53,"Për sukses ndikojnë puna e talenti, por edhe m...",Life &amp; Style,ernis zejnullahu,Ballkani,2024-11-12T09:49:10Z
3,"Pesëdhjetë vjet pas lansimit të këngës “Hello,...",2018-09-26 14:18:01,"Viti 1968, The Doors dhe problemet e Morrisoni...",Showbiz,ernis zejnullahu,Ballkani,2024-11-12T09:49:10Z
4,"Tundni ajkën e ëmbël, shtoni kremin e tundur d...",2018-09-26 14:25:45,Akullore shtëpiake me çokollatë dhe pemë,Life &amp; Style,ernis zejnullahu,Ballkani,2024-11-12T09:49:10Z


## 2. Numri total i rreshtave në dataset

In [9]:
with open('../data/raw/Kosovo-News-Articles.csv', 'r', encoding='utf-8') as f:
    row_count = sum(1 for _ in f)
print(f"Numri i rreshtave (duke përfshirë header): {row_count}")

Numri i rreshtave (duke përfshirë header): 5283843


## 3. EDA mbi sample (50,000 rreshta) — Lorem Ipsum, missing values

In [18]:
df = pd.read_csv('../data/raw/Kosovo-News-Articles.csv', nrows=50000)
print("Shape:", df.shape)
df.info()

lorem_count = df['content'].astype(str).str.contains('Sed ut perspiciatis', na=False).sum()
print(f"\nArtikuj me Lorem Ipsum: {lorem_count} ({lorem_count/len(df)*100:.2f}%)")

print("\nMissing values:\n", df.isnull().sum())

Shape: (50000, 7)
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   content     49999 non-null  str  
 1   date        50000 non-null  str  
 2   title       50000 non-null  str  
 3   category    50000 non-null  str  
 4   author      50000 non-null  str  
 5   source      50000 non-null  str  
 6   fetched_at  50000 non-null  str  
dtypes: str(7)
memory usage: 2.7 MB

Artikuj me Lorem Ipsum: 2 (0.00%)

Missing values:
 content       1
date          0
title         0
category      0
author        0
source        0
fetched_at    0
dtype: int64


## 4. Zbulimi i problemit multi-label

Kolona `category` përmban kombinime kategorish të ndara me `;`. "Ballina" është flamur "homepage", jo kategori tematike — hiqet para analizës.

In [19]:
df['category_clean'] = df['category'].str.replace(r'Ballina;?', '', regex=True).str.strip(';')

print("Bosh pas heqjes (ishin vetëm 'Ballina'):", (df['category_clean'] == '').sum())

df['n_labels'] = df['category_clean'].str.split(';').apply(len)
print("\nShpërndarja e numrit të labels:")
print(df['n_labels'].value_counts())

print("\nKategoritë e pastra (single-label), nga sample 50k:")
print(df[df['n_labels']==1]['category_clean'].value_counts().to_string())

Bosh pas heqjes (ishin vetëm 'Ballina'): 126

Shpërndarja e numrit të labels:
n_labels
1    46710
2     3249
3       41
Name: count, dtype: int64

Kategoritë e pastra (single-label), nga sample 50k:
category_clean
Lajme         26697
Sport          7633
Showbiz        5902
Ballkani       3596
Fun            2431
Video           155
Opinione        152
                126
Intervista       18


## 5. Zbulimi i problemit multi-portal

Dataset-i përmban artikuj nga shumë portale të ndryshme (kolona `source`), secili me taksonominë e vet të kategorive — jo një skemë e vetme konsistente.

In [20]:
df100k = pd.read_csv('../data/raw/Kosovo-News-Articles.csv', nrows=100000)
df100k['category_clean'] = df100k['category'].str.replace(r'Ballina;?', '', regex=True).str.strip(';')

kategorite_6_origjinale = ['Lajme', 'Sport', 'Showbiz', 'Ballkani', 'Fun', 'Life &amp; Style']
df100k['eshte_pastro'] = df100k['category_clean'].isin(kategorite_6_origjinale)

print("'source' për kategoritë e para (nga sample 100k):")
print(df100k[df100k['eshte_pastro']]['source'].value_counts())

'source' për kategoritë e para (nga sample 100k):
source
Ballkani    96088
Name: count, dtype: int64


## 6. Skanim i plotë (5.28M rreshta) — kategoritë sipas portaleve

(Rezultati ruhet te `data/processed/kategori_stats_plote.csv`)

In [21]:
CHUNK_SIZE = 200000
reader = pd.read_csv('../data/raw/Kosovo-News-Articles.csv', chunksize=CHUNK_SIZE, usecols=['category', 'source'])

kategori_stats = {}

for i, chunk in enumerate(reader):
    chunk = chunk.dropna(subset=['category'])
    chunk['category_clean'] = chunk['category'].str.replace(r'Ballina;?', '', regex=True).str.strip(';')
    chunk = chunk[chunk['category_clean'] != '']
    chunk['n_labels'] = chunk['category_clean'].str.split(';').apply(len)
    single = chunk[chunk['n_labels'] == 1]
    
    for cat, group in single.groupby('category_clean'):
        if cat not in kategori_stats:
            kategori_stats[cat] = {'total': 0, 'portale': set()}
        kategori_stats[cat]['total'] += len(group)
        kategori_stats[cat]['portale'].update(group['source'].unique())
    
    print(f"Chunk {i} u përpunua ({(i+1)*CHUNK_SIZE:,} rreshta deri tani)...")

rezultat = pd.DataFrame([
    {'kategoria': k, 'total_shembuj': v['total'], 'nr_portale': len(v['portale']), 'portalet': ', '.join(sorted(v['portale']))}
    for k, v in kategori_stats.items()
])
rezultat = rezultat.sort_values('total_shembuj', ascending=False)

print(f"\nTotal kategori unike (single-label): {len(rezultat)}")
print("\nKategoritë e para 60:")
print(rezultat.head(60).to_string())

rezultat.to_csv('../data/processed/kategori_stats_plote.csv', index=False)

Chunk 0 u përpunua (200,000 rreshta deri tani)...
Chunk 1 u përpunua (400,000 rreshta deri tani)...
Chunk 2 u përpunua (600,000 rreshta deri tani)...
Chunk 3 u përpunua (800,000 rreshta deri tani)...
Chunk 4 u përpunua (1,000,000 rreshta deri tani)...
Chunk 5 u përpunua (1,200,000 rreshta deri tani)...
Chunk 6 u përpunua (1,400,000 rreshta deri tani)...
Chunk 7 u përpunua (1,600,000 rreshta deri tani)...
Chunk 8 u përpunua (1,800,000 rreshta deri tani)...
Chunk 9 u përpunua (2,000,000 rreshta deri tani)...
Chunk 10 u përpunua (2,200,000 rreshta deri tani)...
Chunk 11 u përpunua (2,400,000 rreshta deri tani)...
Chunk 12 u përpunua (2,600,000 rreshta deri tani)...
Chunk 13 u përpunua (2,800,000 rreshta deri tani)...
Chunk 14 u përpunua (3,000,000 rreshta deri tani)...
Chunk 15 u përpunua (3,200,000 rreshta deri tani)...
Chunk 16 u përpunua (3,400,000 rreshta deri tani)...
Chunk 17 u përpunua (3,600,000 rreshta deri tani)...
Chunk 18 u përpunua (3,800,000 rreshta deri tani)...
Chunk 19 u 

## 7. Verifikim: kërkim temash shtesë të mundshme (Teknologji, Auto, etj.)

In [22]:
rezultat = pd.read_csv('../data/processed/kategori_stats_plote.csv')

temat_e_mundshme = ['Teknologji', 'Auto', 'Ligji', 'Moti', 'Kuzhina', 'Kuzhinë', 
                     'Receta', 'Horoskop', 'Arsim', 'Gjykata', 'Video', 'Intervista']

for tema in temat_e_mundshme:
    match = rezultat[rezultat['kategoria'].str.contains(tema, case=False, na=False)]
    if len(match) > 0:
        print(f"\n--- '{tema}' ---")
        print(match[['kategoria', 'total_shembuj', 'nr_portale']].to_string(index=False))


--- 'Teknologji' ---
            kategoria  total_shembuj  nr_portale
     Lajme Teknologji           4997           1
           Teknologji           2488           8
Këshilla Teknologjike            150           1
 Shkencë e Teknologji              5           1

--- 'Auto' ---
 kategoria  total_shembuj  nr_portale
Auto Lajme           6387           1
      Auto           4417           3
  Auto Fun           1303           1
  AutoTech            429           1
 Auto-Tech            279           1
  Autotech            218           1
  AutoMoto             86           1
Auto Sport              1           1

--- 'Ligji' ---
kategoria  total_shembuj  nr_portale
    Ligji            106           1

--- 'Moti' ---
                  kategoria  total_shembuj  nr_portale
                  Motivuese            541           1
                       Moti            239           2
           Moti në Shqipëri              3           1
Moti në Maqedoninë e Veriut              1      

## 8. Verifikim: a është "Lajme" temë koherente apo "catch-all"?

In [23]:
CHUNK_SIZE = 200000
reader = pd.read_csv('../data/raw/Kosovo-News-Articles.csv', chunksize=CHUNK_SIZE, usecols=['category', 'source', 'title'])

shembuj_lajme = []
for chunk in reader:
    chunk = chunk.dropna(subset=['category'])
    chunk['category_clean'] = chunk['category'].str.replace(r'Ballina;?', '', regex=True).str.strip(';')
    match = chunk[chunk['category_clean'] == 'Lajme']
    if len(match) > 0:
        shembuj_lajme.append(match[['source', 'title']])
    if sum(len(s) for s in shembuj_lajme) > 500:
        break

df_lajme = pd.concat(shembuj_lajme, ignore_index=True)
print("Shembuj titujsh nga kategoria 'Lajme', nga portale të ndryshme:\n")
for portal in df_lajme['source'].unique()[:6]:
    print(f"--- {portal} ---")
    print(df_lajme[df_lajme['source']==portal]['title'].head(5).to_string(index=False))
    print()

Shembuj titujsh nga kategoria 'Lajme', nga portale të ndryshme:

--- Ballkani ---
Tavolina në formë të Lamborghini Aventador, me ...
Fat i mirë për PDK-në, që disa do të shkojnë ng...
Thaçi publikon foto tjera nga vizita në veri dh...
 Serbët bllokojnë rrugën në hyrje të Zubin Potokut
Haraqija ‘godet’ Osmanin dhe Avdylin: Sot e mbr...



**Konkluzion:** "Lajme" përmban përzierje heterogjene temash (politikë, lifestyle, siguri, etj.) — nuk është temë e mirëfilltë si Sport/Showbiz, prandaj përjashtohet nga taksonomia finale.